# 11 — SVG postprocessing ablation

**Summary:** Compare multiple SVG postprocessing methods on the same fixed set of raw model outputs. The notebook can either load pre-saved raw generations or generate them from a stored adapter path, then score each method with shared XML parse, renderability, and submission-validity metrics.


#### Colab: mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


#### Install dependencies


In [ ]:
!pip -q install transformers peft tqdm accelerate bitsandbytes datasets trl cairosvg pillow scikit-learn lxml


#### Imports, paths, and device


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

RAW_DIR = PROJECT_DIR / 'data' / 'raw'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_DIR / 'outputs'

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device name:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected.')


#### Parameters

`ADAPTER_DIR` points to stored model weights. Set `USE_STORED_OUTPUTS = True` to skip generation and load a previously exported raw-output table.


In [ ]:
SEED = 42
MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
ADAPTER_DIR = OUTPUTS_DIR / 'models' / 'lora_best_full_train' / 'adapter'

USE_STORED_OUTPUTS = False
RAW_OUTPUTS_PATH = OUTPUTS_DIR / 'analysis' / 'postprocess_ablation_raw_outputs.csv'
SAVE_RAW_OUTPUTS_PATH = RAW_OUTPUTS_PATH

USE_RANKED_DATA = True
RANKED_PATH = PROCESSED_DIR / 'train_ranked.csv'
TRAIN_PATH = RAW_DIR / 'train.csv'
N_SAMPLES = 24
MAX_NEW_TOKENS = 256

USE_PERCENTILE_BUCKETS = True
PERCENTILE_BUCKETS = [
    ('0-10 percentile', 0.00, 0.10),
    ('40-60 percentile', 0.40, 0.60),
    ('80-100 percentile', 0.80, 1.00),
]
N_SAMPLES_PER_BUCKET = 8

N_TEXT_ROWS = 5
PREVIEW_CHARS = 8000

print('ADAPTER_DIR:', ADAPTER_DIR)
print('USE_STORED_OUTPUTS:', USE_STORED_OUTPUTS)
print('RAW_OUTPUTS_PATH:', RAW_OUTPUTS_PATH)
print('USE_RANKED_DATA:', USE_RANKED_DATA, '| USE_PERCENTILE_BUCKETS:', USE_PERCENTILE_BUCKETS)


#### Load evaluation rows

Sample a fixed set of rows once so every postprocessing method is scored on the exact same raw outputs.


In [ ]:
from src.core.dataframe import choose_first_existing
from src.training.prompts import format_svg_instruction_example

if USE_RANKED_DATA and RANKED_PATH.exists():
    eval_source_path = RANKED_PATH
else:
    eval_source_path = TRAIN_PATH

eval_source_df = pd.read_csv(eval_source_path)
print('Loaded eval source:', eval_source_path, 'shape=', eval_source_df.shape)

PROMPT_COL = choose_first_existing(eval_source_df, ['prompt', 'description', 'text'], 'eval_source_df')
TARGET_SVG_COL = None
for c in ('svg', 'svg_code', 'target', 'label', 'ground_truth_svg', 'svg_clean'):
    if c in eval_source_df.columns:
        TARGET_SVG_COL = c
        break

if USE_PERCENTILE_BUCKETS and 'difficulty_percentile' in eval_source_df.columns:
    sample_parts = []
    for bucket_label, low, high in PERCENTILE_BUCKETS:
        if high >= 1.0:
            bucket_df = eval_source_df[(eval_source_df['difficulty_percentile'] >= low) & (eval_source_df['difficulty_percentile'] <= high)].copy()
        else:
            bucket_df = eval_source_df[(eval_source_df['difficulty_percentile'] >= low) & (eval_source_df['difficulty_percentile'] < high)].copy()
        if len(bucket_df) == 0:
            continue
        n_draw = min(N_SAMPLES_PER_BUCKET, len(bucket_df))
        sampled_bucket_df = bucket_df.sample(n=n_draw, random_state=SEED).reset_index(drop=True).copy()
        sampled_bucket_df['difficulty_bucket_label'] = bucket_label
        sample_parts.append(sampled_bucket_df)
    if not sample_parts:
        raise ValueError('No rows available for the configured percentile buckets.')
    sample_df = pd.concat(sample_parts, ignore_index=True)
else:
    n_draw = min(N_SAMPLES, len(eval_source_df))
    sample_df = eval_source_df.sample(n=n_draw, random_state=SEED).reset_index(drop=True).copy()
    sample_df['difficulty_bucket_label'] = sample_df.get('difficulty_bucket_label', 'random_sample')

sample_df['full_prompt'] = sample_df[PROMPT_COL].fillna('').astype(str).apply(
    lambda p: format_svg_instruction_example(p, svg_text=None, include_answer=False)
)
sample_df['id'] = sample_df['id'] if 'id' in sample_df.columns else sample_df.index.astype(str)

print('PROMPT_COL:', PROMPT_COL)
print('TARGET_SVG_COL:', TARGET_SVG_COL)
print('sample size:', len(sample_df))
display(sample_df[['id', PROMPT_COL, 'difficulty_bucket_label']].head())


#### Acquire raw model outputs

Load saved raw generations when available; otherwise generate once from `ADAPTER_DIR`. Raw outputs must remain unsanitized so the postprocessing ablation is meaningful.


In [ ]:
from src.inference.generation import generate_svg_raw_prediction
from src.training.lora.modeling import load_inference_adapter


if USE_STORED_OUTPUTS:
    raw_df = pd.read_csv(RAW_OUTPUTS_PATH)
    print('Loaded stored raw outputs:', RAW_OUTPUTS_PATH, 'shape=', raw_df.shape)
else:
    tokenizer, model = load_inference_adapter(ADAPTER_DIR, MODEL_ID)
    raw_rows = []
    for _, row in sample_df.iterrows():
        prompt_id = row['id']
        prompt_text = str(row[PROMPT_COL])
        target_svg = ''
        if TARGET_SVG_COL is not None and pd.notna(row[TARGET_SVG_COL]):
            target_svg = str(row[TARGET_SVG_COL])
        raw_rows.append(
            {
                'id': prompt_id,
                'prompt': prompt_text,
                'difficulty_bucket_label': row.get('difficulty_bucket_label', 'unknown'),
                'difficulty_percentile': row.get('difficulty_percentile', np.nan),
                'target_svg': target_svg,
                'full_prompt': row['full_prompt'],
                'raw_output': generate_svg_raw_prediction(row['full_prompt'], tokenizer, model, max_new_tokens=MAX_NEW_TOKENS),
            }
        )
    raw_df = pd.DataFrame(raw_rows)
    print('Generated raw outputs shape:', raw_df.shape)
    if SAVE_RAW_OUTPUTS_PATH is not None:
        SAVE_RAW_OUTPUTS_PATH.parent.mkdir(parents=True, exist_ok=True)
        raw_df.to_csv(SAVE_RAW_OUTPUTS_PATH, index=False)
        print('Saved raw outputs to:', SAVE_RAW_OUTPUTS_PATH)

display(raw_df[['id', 'prompt', 'difficulty_bucket_label']].head())


#### Shared helpers for method scoring

These helpers apply each method to `raw_output`, score the resulting SVG, and build the shared aggregate and rescue/harm tables.


In [ ]:
from src.eval.postprocess_ablation import pick_gallery_rows, register_method, score_postprocess_method
from src.eval.postprocess_presets import (
    aggressive_clean,
    conservative_clean,
    hybrid_extract_if_valid_else_repair,
    repair_then_clean,
    truncate_last_nodes_then_clean,
    truncate_non_path_first_then_clean,
    truncate_then_clean_default,
)
from src.inference.postprocess import extract_svg_fragment, sanitize_svg_prediction

METHODS = {}
METHOD_SUMMARIES = {}


## Method 1: Raw Output

Use the decoded model text exactly as produced, with no postprocessing. This is the true baseline and shows how much the later methods help.


In [ ]:
register_method(METHODS, METHOD_SUMMARIES,
    'raw_output',
    lambda text: '' if text is None else str(text),
    'Use the decoded model text exactly as produced, with no postprocessing.'
)


## Method 2: Extract-Only

Run only `extract_svg_fragment()` to isolate the first SVG-looking region and remove obvious wrapper chatter, but do not clean or normalize the SVG.


In [ ]:
register_method(METHODS, METHOD_SUMMARIES,
    'extract_only',
    lambda text: extract_svg_fragment(text),
    'Extract the first SVG fragment but do not run the cleaner.'
)


## Method 3: Current Default Sanitizer

Run `sanitize_svg_prediction()`, which currently applies SVG extraction and then `clean_svg()`. This is the project’s current default baseline.


In [ ]:
register_method(METHODS, METHOD_SUMMARIES,
    'current_default_sanitizer',
    lambda text: sanitize_svg_prediction(text),
    'Apply the project default sanitize pipeline: extract fragment plus clean_svg().'
)


## Method 4: Conservative Cleaner

Apply minimal cleanup aimed at preserving original structure: strip markdown fences and obvious wrapper junk, optionally extract the SVG block, but avoid aggressive canonical rebuilding where possible.


In [ ]:
register_method(METHODS, METHOD_SUMMARIES,
    'conservative_cleaner',
    conservative_clean,
    'Apply light repairs and extraction while preserving as much original structure as possible.'
)


## Method 5: Aggressive Cleaner

Apply stricter normalization intended to maximize submission validity: canonicalize root attributes, strip unsupported content aggressively, and enforce constraints even if fidelity may drop.


In [ ]:
register_method(METHODS, METHOD_SUMMARIES,
    'aggressive_cleaner',
    aggressive_clean,
    'Apply stricter normalization with clean_svg() after wrapper repair.'
)


## Method 6b: Hybrid Extract-If-Valid Else Repair-Then-Clean

Prefer the lighter extraction path when it already yields a valid, renderable SVG, and only fall back to `repair_then_clean()` when extraction alone is not good enough. This tries to preserve the stronger qualitative appearance of lighter methods while keeping the reliability of the stricter repair path.


In [ ]:
register_method(METHODS, METHOD_SUMMARIES,
    'hybrid_extract_if_valid_else_repair',
    hybrid_extract_if_valid_else_repair,
    'Keep extracted SVG when it is already valid and renderable; otherwise fall back to repair_then_clean().'
)


## Method 6: Repair-Then-Clean

Try lightweight structural repairs before the main cleaner, such as removing code fences, clipping trailing junk, or adding a missing closing `</svg>` when the output is obviously truncated, then run `clean_svg()`.


In [ ]:
register_method(METHODS, METHOD_SUMMARIES,
    'repair_then_clean',
    repair_then_clean,
    'Attempt lightweight wrapper repair before applying clean_svg().'
)


## Method 7: Truncation Strategy Variants

Compare alternative ways to handle oversized SVGs under the `16000`-character cap. This section includes multiple truncation-oriented sub-methods so you can see whether different removal priorities change renderability or validity.


In [ ]:
register_method(METHODS, METHOD_SUMMARIES,
    'truncate_then_clean_default',
    truncate_then_clean_default,
    'Use the project default cleaner as the baseline truncation strategy.'
)
register_method(METHODS, METHOD_SUMMARIES,
    'truncate_last_nodes_then_clean',
    truncate_last_nodes_then_clean,
    'Trim oversized SVGs by repeatedly removing the last child node before cleaning.'
)
register_method(METHODS, METHOD_SUMMARIES,
    'truncate_non_path_first_then_clean',
    truncate_non_path_first_then_clean,
    'Trim oversized SVGs by removing non-path nodes before dropping remaining trailing nodes.'
)

display(pd.DataFrame({'method_name': list(METHODS.keys()), 'summary': [METHOD_SUMMARIES[k] for k in METHODS]}))


#### Score all methods

Apply each method to the same raw-output dataframe, then compute aggregate metrics, rescue/harm counts, and a long per-example comparison table.


In [ ]:
scored_dfs = [
    score_postprocess_method(name, fn, METHOD_SUMMARIES[name], raw_df)
    for name, fn in METHODS.items()
]
all_results_df = pd.concat(scored_dfs, ignore_index=True)

aggregate_df = (
    all_results_df.groupby('method_name', as_index=False)
    .agg(
        xml_parse_rate=('xml_parse_ok', 'mean'),
        render_rate=('render_ok', 'mean'),
        submission_valid_rate=('submission_valid', 'mean'),
        svg_open_rate=('has_svg_open', 'mean'),
        svg_close_rate=('has_svg_close', 'mean'),
        avg_pred_char_len=('pred_char_len', 'mean'),
        avg_path_count=('path_count', 'mean'),
        changed_rate=('changed_output', 'mean'),
    )
    .sort_values(['submission_valid_rate', 'render_rate', 'xml_parse_rate'], ascending=False)
    .reset_index(drop=True)
)

rescue_harm_df = (
    all_results_df.assign(
        rescued_submission=(~all_results_df['raw_valid_submission']) & (all_results_df['submission_valid']),
        harmed_submission=(all_results_df['raw_valid_submission']) & (~all_results_df['submission_valid']),
        rescued_render=(~all_results_df['raw_valid_submission']) & (all_results_df['render_ok']),
    )
    .groupby('method_name', as_index=False)
    .agg(
        rescued_submission_count=('rescued_submission', 'sum'),
        harmed_submission_count=('harmed_submission', 'sum'),
        rescued_render_count=('rescued_render', 'sum'),
    )
)

display(HTML('<h3>Aggregate method comparison</h3>'))
display(aggregate_df)
display(HTML('<h3>Rescue / harm summary</h3>'))
display(rescue_harm_df)
display(HTML('<h3>Per-example long table (head)</h3>'))
display(all_results_df.head())


#### Qualitative review tables

Use these slices to inspect rescue examples, harmful changes, XML-parse-only failures, and oversized SVG behavior before deciding which methods are worth promoting into reusable code.


In [ ]:
rescues_df = all_results_df[(~all_results_df['raw_valid_submission']) & (all_results_df['submission_valid'])].copy()
harms_df = all_results_df[(all_results_df['raw_valid_submission']) & (~all_results_df['submission_valid'])].copy()
xml_but_not_render_df = all_results_df[(all_results_df['xml_parse_ok']) & (~all_results_df['render_ok'])].copy()
oversize_df = all_results_df[(all_results_df['pred_char_len'] > MAX_SVG_CHARS) | (~all_results_df['constraint_within_char_limit'])].copy()

display(HTML('<h3>Best rescue examples</h3>'))
display(rescues_df[['method_name', 'id', 'difficulty_bucket_label', 'pred_char_len', 'raw_output', 'postprocessed_svg']].head(N_TEXT_ROWS))

display(HTML('<h3>Examples harmed by postprocessing</h3>'))
display(harms_df[['method_name', 'id', 'difficulty_bucket_label', 'raw_output', 'postprocessed_svg']].head(N_TEXT_ROWS))

display(HTML('<h3>XML parses but rendering fails</h3>'))
display(xml_but_not_render_df[['method_name', 'id', 'difficulty_bucket_label', 'postprocessed_svg']].head(N_TEXT_ROWS))

display(HTML('<h3>Examples exceeding char/path constraints before or after processing</h3>'))
display(
    all_results_df[
        ['method_name', 'id', 'pred_char_len', 'path_count', 'constraint_within_char_limit', 'constraint_within_path_limit']
    ].sort_values(['pred_char_len', 'path_count'], ascending=False).head(N_TEXT_ROWS)
)


#### Rendered SVG gallery by method

Render a few postprocessed SVG outputs for each method so you can quickly compare qualitative behavior, not just aggregate metrics. The gallery prioritizes renderable examples for each method when available.


In [ ]:
import matplotlib.pyplot as plt

from src.svg.rendering import render_svg_or_none

N_RENDER_PER_METHOD = 3


for method_name in aggregate_df['method_name']:
    method_df = all_results_df[all_results_df['method_name'] == method_name].copy()
    gallery_df = pick_gallery_rows(method_df, N_RENDER_PER_METHOD)

    display(HTML(f'<h3>{method_name}</h3>'))
    summary_txt = METHOD_SUMMARIES.get(method_name, '')
    if summary_txt:
        display(HTML(f'<p>{summary_txt}</p>'))

    if len(gallery_df) == 0:
        display(HTML('<p>No examples available.</p>'))
        continue

    fig, axes = plt.subplots(len(gallery_df), 2, figsize=(12, 4 * len(gallery_df)))
    if len(gallery_df) == 1:
        axes = np.array([axes])

    for row_idx, (_, rec) in enumerate(gallery_df.iterrows()):
        ax0, ax1 = axes[row_idx]
        ax0.axis('off')
        ax1.axis('off')

        prompt_txt = str(rec.get('prompt', '') or '').strip()
        prompt_preview = prompt_txt[:220] + ('...' if len(prompt_txt) > 220 else '')
        ax0.text(
            0.02,
            0.98,
            f"id = {rec['id']}\nrender_ok = {bool(rec['render_ok'])}\nxml_parse_ok = {bool(rec['xml_parse_ok'])}\n\n{prompt_preview}",
            ha='left',
            va='top',
            fontsize=9,
            wrap=True,
            transform=ax0.transAxes,
        )
        ax0.set_title('Prompt / metadata')

        img = render_svg_or_none(rec['postprocessed_svg'])
        if img is not None:
            ax1.imshow(img)
        else:
            ax1.text(0.5, 0.5, 'Rendered preview failed', ha='center', va='center')
        ax1.set_title('Postprocessed SVG render')

    plt.tight_layout()
    plt.show()
